<a href="https://colab.research.google.com/github/codeishitech/Predicting-Primary-Visual-Cortex-V1-Neural-Responses/blob/main/notebooks/01_data_and_response.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import scipy.io
mat = scipy.io.loadmat('01.mat')
print(list(mat.keys()))

In [ ]:
print(type(mat['RF_SPATIAL']))
print(mat['RF_SPATIAL'].shape)
print(mat['RF_SPATIAL'][:10])

In [ ]:
print("resp_train shape:", mat['resp_train'].shape)
print("images shape/type:", mat['images'].shape, type(mat['images']))
print("INDCENT shape:", mat['INDCENT'].shape)
print("Number of centered neurons:", mat['INDCENT'].sum())

In [ ]:
#Raw data loaded, response matrix built [neurons × images]
import numpy as np
natural_idx = slice(0,540)
cen_neurons = mat['INDCENT'].flatten() == 1
#spike counting window, skipping the first 50 ms which is the latency period or delay in propagation
start = 50
spike_counts = mat['resp_train'][:,natural_idx,:,start:].sum(axis=3)
mean_response = spike_counts.mean(axis=2)
mean_centered_response = mean_response[cen_neurons, : ]
print("Response matrix shape:", mean_centered_response.shape)

In [ ]:
#Build matching image matrix [images × pixels/features]
natural_images = mat['images'][0,:540]
print("Natural images shape:", natural_images.shape)
print(type(natural_images), natural_images[0].shape if hasattr(natural_images[0], 'shape') else "no shape attr")
print(natural_images[4])

In [ ]:
import torch
import torchvision.transforms as T
natural_images = mat['images'][0,:540]
img_stck = np.stack([np.array(img) for img in natural_images])
print("Stacked shape:", img_stck.shape)
print("Pixel value range:", img_stck.min(), img_stck.max())
print(img_stck[0])

In [ ]:
import torchvision.models as models
import torch.nn as nn
vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
vgg.eval()
print(vgg.features)

In [ ]:
#preprocessing the image:
# VGG-16 expects: 224x224, 3-channel, normalized with ImageNet stats
preprocess = T.Compose([
    T.ToPILImage(),
    T.Resize((224, 224)),
    T.Grayscale(num_output_channels=3),  # duplicate grayscale into 3 channels
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Convert one test image to check it works
test_img = img_stck[0].astype(np.uint8)
test_tensor = preprocess(test_img)
print("Preprocessed shape:", test_tensor.shape)